# Clase 13: Regresión

**MDS7202: Laboratorio de Programación Científica para Ciencia de Datos**

**Profesor: Diego Cortez**


## Objetivos de la Clase

Conceptual
- Comprender lo que es aprendizaje supervisado
- Framework general: objetivos y métricas
- Overfitting y regularización
- El problema de la regresión
- Métricas de regresión
  
Aplicación
- Regresión lineal
- Regresión lineal regularizada: Ridge y LASSO
- Train test split
- Trade-off sesgo-varianza
- Pipeline predictivo
- MLFlow

---

## Machine learning en general

<br>

<div align='center'>
<img src="https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/machine_learning.png?raw=true" alt="Panorama General ML: Clasificación supervisada, No supervisada y Aprendizaje Reforzado binario" width=700/>
</div>



#### _Aprendizaje de Máquinas_

En general, machine learning es el conjunto de técnicas en que una máquina o algoritmo puede ***aprender*** un patrón en base a datos. Esto ya lo hemos visto aplicado en 2 grupos de prácticas:
- **Preprocesamiento y Feature engineering.** Procesadores necesitan 'aprender' qué operaciones realizar a una columna para preprocesarla. Ej., Mínimos y máximos, media, cantidad de categorías, etc.
- **Aprendizaje no supervisado.** Un algoritmo aprende agrupaciones, relaciones entre variables, transformaciones que revelan patrones subyacentes en los datos.

En ambos casos, el modelo _aprende_ algo en base a los datos de forma que después pueda _aplicar_ lo aprendido a datos nuevos, y determinar:
- Si el nuevo dato presenta algún patrón observado previamente
- Si el nuevo dato corresponde a algún grupo

También sabemos que para esto el modelo realiza **Operaciones matemáticas** con los atributos para obtener su respuesta de acuerdo a **reglas** determinadas por el **algoritmo del modelo**. El algoritmo define cómo encontrar los **parámetros** del modelo que permitirán operar los atributos para obtener el resultado que minimíce una **métrica**. Un pseudocódigo para entender esto podría ser:


```python
# Set de reglas en que se operan los atributos
import AlgoritmoMachineLearning 

# Opciones no aprendibles que definen variaciones al algoritmo
hiperparametros = {
    "hiperparametro_1": valor_1
}

# Modelo: instancia del algoritmo con los hiperparámetros el cual puede aprender de los datos
modelo = AlgoritmoMachineLearning(**hiperparametros)

print(modelo._parametros) # Imprime None. El modelo comienza en blanco

# Entrenar modelo con datos y objetivo a aprender (opcional)
if AlgoritmoMachineLearning.es_supervisado:
    modelo._parametros = modelo.fit(datos, columna_objetivo)
else:
    modelo._parametros = modelo.fit(datos)

# Modelo puede predecir lo aprendido con datos nuevos. Sus parámetros reflejan el aprendizaje
prediccion = modelo.predict(datos_nuevos, modelo._parametros)
```

## Aprendizaje Supervisado

En aprendizaje supervisado se basa en trabajar con datasets cuyas observaciones son conjuntos de vectores con distintas **características** que describen a algún objeto más una **etiqueta/valor real** la cuál asigna una clase o valor a cada objeto:

- $X$: vectores de atributos de los datos
- $Y$: etiqueta o valor a predecir

Cuando el valor a predecir es un(a): 

- Categoría/Etiqueta, el problema que se resuelve se denomina **Clasificación**.
- Valor real, el problema que se resuelve se denomina **Regresión**.


Las etiquetas pertenecen a un número finito de clases.


Por ejemplo, el caso que estemos describiendo a una persona, el vector puede contener las siguentes **features/características**:

- su altura en cm, 
- edad, 
- peso en kg, 
- residencia, 
- etc...

Mientras que la **etiqueta** puede ser si la persona *quiere o no contratar un servicio de internet*: $\{ True, False\}$ 

In [35]:
import pandas as pd

# ejemplo clasificación
pd.DataFrame(
    [[177, 43, 72, "Maipú", True], [160, 16, 60, "Pudahuel", False]],
    columns=["Altura", "Edad", "Peso", "Residencia", "Posible cliente?"],
)

,Altura,Edad,Peso,Residencia,Posible cliente?
0,177,43,72,Maipú,True
1,160,16,60,Pudahuel,False


Como también lo que está dispuesto a gastar en el plan.

In [36]:
# ejemplo regresión
pd.DataFrame(
    [[177, 43, 72, "Maipú", 55000], [160, 16, 60, "Pudahuel", 0]],
    columns=["Altura", "Edad", "Peso", "Residencia", "Cuánto está dispuesto a pagar?"],
)

,Altura,Edad,Peso,Residencia,Cuánto está dispuesto a pagar?
0,177,43,72,Maipú,55000
1,160,16,60,Pudahuel,0


El objetivo final del aprendizaje supervisado es crear modelos que permiten **asignar de forma automática categorías o valores a observaciones nuevas**. 

En términos prácticos, dado una nueva observación representada por un vector de características, el modelo generado debe ser capaz de asignar una etiqueta a dicha observación.


### Un buen ajuste

El modelo construido debe **generalizar**, es decir, debe ser capaz de realizar predicciones correctas en nuevas observaciones. Para el ejemplo de clasificación, podemos considerar que el modelo generado está separando los datos por clases a través de un *límite de decisión*. Este límmite de decisión podría ser muy ajustado o más holgado, y muchas veces de esto depende qué tan bien puede generalizar

<div align='center'>
<img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/overfitting.png?raw=true' witdh=400/>

</div>

<div align='center'>
<a href='https://www.researchgate.net/figure/Example-of-overfitting-in-classification-a-Decision-boundary-that-best-fits-training_fig1_349186066'>Ejemplo de *Overfitting* en researchgate</a>
</div>

> **Pregunta ❓:** En la imágen, cuál de las 2 clasificación es mejor? Qué ventajas y desventajas tiene cada una?

#### Sobreajuste desde un enfoque generativo

Realizar machine learning implica considerar que la etiqueta buscada de alguna forma depende de los datos. Es decir, se asume el siguiente modelo:

```
variable_objetivo = modelo(datos)
```

o matemáticamente

$$Y=f(X)$$

Es decir, que la variable que se busca predecir es una función de los datos. Ciertamente, podríamos definir una función matemática que predice correctamente la etiqueta en base a los datos. Sin embargo, para entrenar un buen modelo debemos pensar en _por qué_ una variable objetivo podría predecirse en función de los datos? La respuesta es que existe un **fenómeno real** que produce relaciones entre diferentes variables, el cual buscamos modelar de alguna forma.

$$
\begin{align}
Z→X\\
Z→Y
\end{align}
$$

Donde $Z$ es el fenómeno real subyacente, $X$ son los datos observados e $Y$ es la variable a predecir. En esta relación, puede o no existir una relación causal entre $X$ e $Y$, pero sí existe un fenómeno subyacente que define ambos y por lo tanto permite definir $Y$ en función de $X$. Sin embargo, cabe notar 2 fenómenos:

- $X$ e $Y$ no necesariamente son **todas** las variables relevantes producidas por $Z$. Puede haber variables no observadas
- En el fenómeno $Z$, pueden existir procesos aleatoreos, por lo que incluso si se tuvieran todas las variables relevantes, existe incertidumbre y no es posible predecir de forma perfecta

Por esto, el modelo real que se busca obtener tiene en realidad la siguiente forma

$$Y = f(X) + \epsilon$$

Donde
- $X$ Son los datos disponibles
- $Y$ Es la etiqueta a predecir
- $f$ es el modelo predictivo
- $\epsilon$ es el error de predicción o incertidumbre

¿Qué tanto se puede acercar la predicción a la etiqueta real? Tanto como la relación real del **fenómeno subyacente** defina entre $X$ e $Y$. Si el modelo se ajusta de forma extremadamente precisa puede estar capturando no solo la relación real, sino también el ruido específico de esos datoses. En este caso, se habla que el modelo está **sobreajustado** ya que la relación entre $Y$ y $\epsilon$ no se verá reflejada en datos nuevos.


#### Ejemplo

Por ejemplo, en el caso de uso del **pronóstico del tiempo**, tenemos:
- $Z$ = estado atmosférico
  - Humedad
  - Presión
  - Temperatura
  - Dinámicas del aire

En este caso el fenómeno subyacente $Z$ implica que existen dinámicas generativas que producen que estas variables interactúen entre sí y se influencien, afectando sus valores en el futuro. En base a esto, podemos tener el siguiente problema de predicción.
- $X$ = datos disponibles
  - Densidad de nubes
  - Lecturas de humedad
  - Presión barométrica
  - Lecturas de termómetro
  - Fecha
  - Valor del dólar
  - Tasa de desempleo
  - Nivel de congestión de tráfico
  - Consumo elétrico
  - Nivel de actividad de twitter
- $Y$ = objetivo
  - Precipitaciones (mm) 

En este caso, variables como la densidad de nubes, humedad, presión y temperatura en días anteriores podrían ser un reflejo de fenómenos atmosféricos que influyen en la probabilidad de lluvia del futuro, y la fecha influye de qué forma estos influyen. Sin embargo, encontramos variables que _probablemente_ no influyen en la probabilidad de lluvia, como el valor del dólar, tasa de desempleo, etc... Si bien podría existir una relación que refleje un fenómeno subyacente (ej., mayor tráfico = mayor smog, consumo eléctrico depende del nivel de obscuridad, etc.) probablemente esta relación es muy débil, por lo que estas variables principalmente **ensucian** nuestro modelo. 

Adicionalmente, existen variables que no estamos observando y que serían útiles. Como las precipitaciones en días anteriores, presión y temperatura en lugares cercanos, fenómenos externos como la niña o el niño, etc. 

Toda esta información faltante es necesaria para tener un modelo confiable. Como no la tenemos, el modelo intenta predecir $Y$ en base al $X$ disponible, proceso en el cual puede capturar relaciones inexistentes: encontrar una relación entre el valor del dolar y la lluvia que en realidad es aleatorea, o buscar predecir exactamente la lluvia en base a lo que tiene. En ese caso el modelo estará **sobreajustado**.

Un modelo correctamente ajustado, predicirá todo lo de $Y$ que es realmente reflejado por $X$. Es decir, no reflejará una relación perfecta entre $X$ e $Y$, sino aquello que sí pueda encontrarse en datos nuevos.

> **Nota:** No existe forma en que el modelo 'sepa' qué tan bien ajustado está, o si la relación es o no extendible a datos nuevos. La única forma confiable de obtener esto es evaluar el desempeño del modelo siempre en datos no utilizados para entrenar, mientras se introducen **regularizaciones** en el algoritmo que se saben que reducen el sobreajuste de los modelos.

#### ⚖️ Tradeoff Bias-Varianza

Existen 2 fuentes de errores que hacen que los modelos generalicen correctamente más allá de sus datos de entrenamiento: El error de un modelo puede descomponerse en tres componentes:

- Sesgo (bias): error por suposiciones incorrectas del modelo
- Varianza (variance): error por sensibilidad a los datos de entrenamiento
- Ruido: incertidumbre inherente al problema

Existe un compromiso entre sesgo y varianza:

- Modelos simples → alto sesgo, baja varianza
- Modelos complejos → bajo sesgo, alta varianza

**🔁 Relación con sobreajuste**
- Sobreajuste: baja bias, alta varianza
- Subajuste: alta bias, baja varianza

Un buen modelo logra un equilibrio entre ambos.

<div align='center'>
<img src='https://substackcdn.com/image/fetch/$s_!DsYV!,w_1456,c_limit,f_webp,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F4e91bc92-8b4f-4545-8433-54c0ef168d9a_700x358.png' witdh=400/>

</div>


### Framework general de aprendizaje supervisado

Cuando realizamos aprendizaje supervisado, buscamos que en base a un **algoritmo**, un **modelo** aprenda las transformaciones matemáticas que permiten acercarse lo más posible a una etiqueta o valor deseado. Para lograr esto, se busca optimizar una **métrica**


#### Métricas


Las métricas son mediciones de qué tan bien se está desempeñando el modelo, o en otras palabras, qué tanto se parece la etiqueta predicha por el modelo de machine learning a la etiqueta real. Estas pueden ser métricas de **similitud** o métricas de **error**.

**Métricas de similitud**

Son métricas que indican *qué tanto se parece* la predicción del modelo a la etiqueta real. Mientras más alta la métrica, mejor el desempeño del modelo, por lo tanto se busca **maximizarlas** en el entrenamiento. Ejemplos:
- Clasificación: Accurracy, F1-score, ROC-AUC
- Regresión: $R^2$, 

**Métricas de error, o Loss**

Son métricas que indican *qué tanto se distancia* la predicción del modelo de la etiqueta real. Mientras más bajas, mejor el desempeño del modelo ya que indican que este tiene menos error, por lo tanto se busca **minimizarlas** en el entrenamiento. Ejemplos:
- Clasificación: Entropía cruzada
- Regresión: Error cuadrático medio (MSE), Error absoluto medio (MAE), RMSE

El _cómo_ minimizo o maximizo la métrica modificando los parámetros depende del algoritmo usado

#### Pipeline de aprendizaje supervisado

La siguiente lista muestra las etapas que debería cumplir un algoritmo de aprendizaje supervisado clásico (i.e., no red neuronal)

1. **Feature Engineering y Preprocesamiento**: Recolectar y preparar los datos.
2. **Entrenar** un algoritmo de clasificación/regresión usando los datos.
3. **Evaluar** qué tan bien clasifica el modelo generado.
4. **Optimizar los modelos** modificando sus hiperparámetros.

### Cómo determinar que algorimo utilizar 

Lo más importante es la **capacidad predictiva** del modelo.
Sin embargo, también hay otros factores muy relevantes que determinarán que algoritmo predictivo utilizar: 

**Eficiencia**: 
  - ¿Qué tanto se está demorando mi modelo en entrenar? 
  - ¿Y en predecir? 
  - ¿Es eficiente en memoria? 
  - ¿Debe almacenar el dataset de entrenamiento para funcionar?
  - ¿Es posible usarlo en tiempo real para algún tipo de solución online?
  
**Número de Features y Ejemplos Requeridos**: 
  - ¿Cuántos datos o features son requeridos para entrenar el modelo?
  - ¿Es compatible con la cantidad que dispongo?
  - ¿El tipo de features (i.e., categorícas, numéricas, combinación de ambas, etc...) es compatible con el algoritmo?
  
**Explicabilidad**: 
  - ¿Puedo explicar por qué el modelo está clasificando/regresionando de la manera que lo hace? 
  
***Fairness***: 
  - ¿Mi modelo es injusto con respecto a algún grupo social?

## Regresión

Regresión es el problema que consiste en generar modelos que sean capaces de predecir valores reales los cuales son comúnmente llamados target.
Uno de los ejemplos más clásicos (que incluso usamos al comienzo del curso) es, dados los atributos de una vivienda, construir modelos que permitan asignarle un precio de forma automatizada. 

<div align='center'>
<img src="https://datahacker.rs/wp-content/uploads/2022/07/Picture3-scaled.jpg" style="width: 600px;"/>

</div>

En este caso, un modelo de regresión buscará en base a los datos existentes (tamaño de la casa, tamaño del terreno, año, habitaciones, material, etc.) buscará predecir lo más fidedignamente posible el precio de la casa. O mejor dicho, buscará **optimizar** alguna de las **métricas de regresión** entre la predicción del modelo y el valor real del precio.




### Métricas de Rendimiento de Regresión


#### MSE - Mean Squared Error 

Una de las métricas más usadas en regresión. Cuando el modelo minimiza el MSE, la predicción se acerca a **la media** esperada de $Y$ en función a $X$.

 $$MSE = \frac{1}{N}\sum_{i=1\dots n} (y_i - \hat{y_i})^2$$


#### RMSE - Root Mean Squared Error

Esta variante consiste en calcular la raíz cuadrada de MSE. La idea principal de esta métrica es que el error sea interpretable ya que queda con la unidad que se está prediciendo. 

 $$RMSE = \sqrt{\frac{1}{N}\sum_{i=1\dots n} (y_i - \hat{y_i})^2}$$
 
 
#### MAE - Mean Absolute Error

Esta variante no calcula el error de forma cuadrática, si no que lo hace simplemente con un valor absoluto

 $$MAE = \frac{1}{N}\sum_{i=1\dots n} |y_i - \hat{y_i}|$$
 
 

#### MAE - Median Absolute Error

Esta variante, muy similar a la anterior, calcula la mediana en vez de la media de las diferencias.

 $$MedAE = median(|y_1 - \hat{y_1}|, \dots, |y_n - \hat{y_n}|)$$
 
La particularidad de esta formulación es que, a diferencia de la anterior, es resistente a outliers gracias al su cálculo basado en la media.

#### Coeficiente de Determinación R²

El Coeficiente de Determinación R² es un puntaje que representa la proporción de varianza de los valores predichos por el modelo con respecto a la los target reales.


$$R^2 (y, \hat{y}) = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y_i})^2}{\sum_{i=1}^n (y_i - \bar{y})^2}$$

En donde $y_i$ es el valor real del target de un ejemplo $x_i$, $\hat{y_i}$ es el valor predicho por el modelo y $\bar{y}$ es la media de los targets reales ($\bar{y} = \frac{1}{n}\sum_{i=1}^{n} y_i$).
$\sum_{i=1}^{n} (y_i - \hat{y_i})^2$ también se conoce como suma de cuadrática de los residuos.


<div align='center'>
<br>
<img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/20_regresion_lineal/r2.png?raw=true' width=600 />
<br>
<div> 
    Coeficiente de Determinación en <a href='https://en.wikipedia.org/wiki/Coefficient_of_determination'>Wikipedia (Inglés)</a>.
    <br>
En esta infografía, se explica el R² como 1 menos la suma del area los cuadrados azules dividida por la suma de los cuadrados rojos.
    
<br>    
</div>
    
<br>

Puede ser útil pensar que lo que se está comparando es un modelo baseline que siempre predice la media (el de la izquierda) con respecto a un modelo entrenado. Luego, la proporción indica que tanto mejora el modelo entrenado con respecto al baseline. 


El mejor puntaje posible es 1 (los mejores están cercanos a este) y este puede ser negativo (con modelos extremadamente malos).


### Underfitting y Overfitting

Errores de entrenamiento o **Underfitting**. 
- Malos resultados sobre los datos de entrenamiento
- El clasificador no tiene capacidad de aprender el patrón.

Errores de generalización o **Overfitting**. 
- Malos resultados sobre datos nuevos 
- El modelo se hace demasiado específico a los datos de entrenamiento. 


<div align='center'>
<img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/tipos_fit.png?raw=true' width=800/>
</div>

<div align='center'>
    Fuente: The Hundred-Page Machine Learning Book.
</div>

### Modelo base: regresión lineal

Sea un conjunto de ejemplos etiquetados (i.e., vectores de $D$ características con sus respectivos targets)
$\{(\mathbf{x}^{(i)}, y^{(i)}) \}_{i=1}^{N}$ respectivamente y con N la cantidad de ejemplos. $x_j^{(i)}$ con $j = 1, \dots, D$ es el vector de características que describe al ejemplo $i$


Una Regresión Lineal consiste en la construcción de un modelo $f_{w,b} = wx + b$ orientado a predecir el target usando una combinación lineal de las características de cada ejemplo de la forma: 

$$f_{\mathbf{w}, b}(x) = \mathbf{wx} + b$$

$$f_{\mathbf{w}, b}(x) = w_{0} x_0 + w_1 x_1 + \dots + w_n x_n$$

En donde $\mathbf{w}$ es el vector de parámetros de tamaño $D$ (que define una pendiente en un hiperplano) y $b$ el intercepto. 
En este caso, se dice que el modelo construido $f_{\mathbf{w}, b}$ está *parametrizado* por $\mathbf{w}$ y $b$.

$f_{\mathbf{w}, b}(x)$ también se le denomina $\hat{y_i}$, el cuál es simplemente es el valor predicho por el modelo.


Cada posible combinación de los parámetros generará una regresión distinta. Por ende, la idea es encontrar el conjunto de parámetros que más se ajusten a los datos 


La regresión lineal es de los modelos más simples de Machine Learning. 
- Modelo extremadamente simple, ápto para tareas con alta carga computacional
- Baja varianza, alto sesgo.
- Variantes con mayor regularización: Ridge y LASSO
- Es de los pocos modelos con **solución cerrada** por lo que es completamente reproducible. 
- Es interpretable, a través de su vector de características
- Su vector de características, si las variables están escaladas, es un indicador de la importancia de las variables

## Aplicación práctica

### Auto-mpg dataset

Para ejemplicar una Regresión Lineal, usaremos el dataset [Auto-mpg dataset](https://www.kaggle.com/uciml/autompg-dataset), el cual, dado distintas características de autos antiguos, intenta predecir el consumo de galones por milla (miles per gallon, mpg)

![Auto-mpg dataset](https://storage.googleapis.com/kaggle-datasets-images/1489/2667/d7895dcd2db5e0cfda19c3edc2f2d410/dataset-cover.jpg)

<div align='center'>
Fuente: Competencia en Kaggle.
</div>


Son 398 autos. Los atributos que los describen son:

    - mpg: continuous
    - cylinders: multi-valued discrete
    - displacement: continuous
    - horsepower: continuous
    - weight: continuous
    - acceleration: continuous
    - model year: multi-valued discrete
    - origin: multi-valued discrete
    - car name: string (unique for each instance)

In [3]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('https://raw.githubusercontent.com/MDS7202/MDS7202/refs/heads/main/recursos/2023-01/20_regresion_lineal/auto-mpg.csv')
df = df.dropna()
df

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino
...,...,...,...,...,...,...,...,...,...
393,27.0,4,140.0,86,2790,15.6,82,1,ford mustang gl
394,44.0,4,97.0,52,2130,24.6,82,2,vw pickup
395,32.0,4,135.0,84,2295,11.6,82,1,dodge rampage
396,28.0,4,120.0,79,2625,18.6,82,1,ford ranger


In [4]:
fig = px.scatter(
    df,
    x='displacement',
    y='mpg',
    template='plotly_white',
    color='mpg',
    hover_name='car name',
    trendline='ols', # este parámetro permite calcular rápidamente la regresión sobre x e y.
    title=
    "Cilindrada de Distintos Automóviles con Respecto a su Rendimiento (mpg)<br>"
    "<sub>La linea representa una regresión lineal calculada sobre ambas variables.</sub>"
)
fig.show()

In [5]:
px.scatter_matrix(df.drop(columns=['car name', 'origin']),
                  title='Scatter Matrix mpg Dataset',
                  height=800,
                  template='plotly_white',
                  color='mpg',
                  color_continuous_scale='Viridis',
                  hover_name='mpg',
                 ).update_traces(diagonal_visible=False, showupperhalf=False)

In [6]:
features = df.drop(columns=['car name', 'mpg'])
target = df['mpg']

features.columns

Index(['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration',
       'model year', 'origin'],
      dtype='str')

En el caso de usar todos los atributos, cada observación $\mathbf{x_i}$ estará compuesta por `'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model year', 'origin'` y etiquetada con el target a predecir `mpg` (millas por galón). Al igual que en la clasificación, para entrenar el modelo se separa el dataset en entrenamiento y prueba.

En el bloque de código anterior, eliminamos una variable que estimamos no influye en la predicción, que es `car name`. Por lo expuesto en la sección conceptual, esto ayuda a entrenar modelos más generalizables. Para asegurar generalización, debemos `evaluar el modelo en datos no usados para entrenar.`

---

## Holdout

Consiste en particionar nuestro dataset en conjuntos de:

- **Training**: conjunto que se utiliza para **entrenar** el modelo.
- **Testing**: datos que se usa para **evaluar** qué tan bien predice el modelo (a través de las métricas de evaluación). 


Comunmente se dividen en proporción $2/3$ y $1/3$ del dataset respectivamente. Sin embargo, todo depende de la cantidad de datos que se posean: si se tiene millones de ejemplos, quizas puede dividirse en 95% train, 5% test sin problemas. 


La evaluación puede variar mucho según las particiones escogidas: 

- Training pequeño -> modelo sesgado, 
- Testing pequeño -> evaluación poco confiable.


Esta ténica se puede **Random Subsampling** para seleccionar aleatoriamente las observaciones de cada uno de estos conjuntos.

Para ejecutar todo esto usaremos `train_test_split`. Veamos algunos de sus parámetros:

- `test_size = 0.33` - indica el tamaño del test de evaluación.
- `shuffle = True` - indica que ejecutaremos Random Subsampling.
- `stratify = labels` - intenta manetener la distribución de clases original en ambos conjuntos.


### Validation set:

Cuando se desea realizar una búsqueda de los mejores algoritmos y sus hiperparámetros, el dataset puede ser dividido en 3:


- **Training**: Se utiliza para entrenar los modelos.
- **Validation**: Se utiliza para seleccionar el mejor modelo al ir variando sus hiperparámetros.
- **Testing**: Se utiliza para evaluar el modelo previo a ser entregado o puesto en producción. Esta evaluación solo se hace sobre el modelo final.


En este caso, en nuestro primer entrenamiento comenzaremos solo con sets de entrenamiento y test, con una división de $70\%$ y $30\%$ respectivamente.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test  = train_test_split(
    features, target, shuffle=True, test_size=0.3, random_state=33
)

In [8]:
X_train.head(3)

,cylinders,displacement,horsepower,weight,acceleration,model year,origin
52,4,88.0,76,2065,14.5,71,2
149,4,120.0,97,2489,15.0,74,3
394,4,97.0,52,2130,24.6,82,2


In [9]:
X_train.shape

(278, 7)

In [10]:
y_train.head(3)

52     30.0
149    24.0
394    44.0
Name: mpg, dtype: float64

In [11]:
y_train.shape

(278,)

### Preprocesamiento y pipeline predictivo

Como visto en clases anteriores, para una confección adecuada del modelo es necesario preprocesar los datos, y esto puede ser puesto en un solo pipeline de preprocesamiento. Una ventaja de utilizar pipelines, es que este puede contener tanto el preprocesamiento como el modelo, de forma tal que el pipeline contiene el proceso completo de tratamiento de datos.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

preprocessing = ColumnTransformer(
    [
        ("standard", StandardScaler(), ["displacement", "weight", "acceleration"]),
        ("ohe", OneHotEncoder(drop='first'), ["origin"]),
        ("ordinal", OrdinalEncoder(), ["cylinders", "model year"]),
    ]
)

Entrenaremos un modelo de **regresión lineal** para predecir nuestra etiqueta, utilizando la clase `LinearRegression` del módulo `linear_model` de sklearn

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

pipe = Pipeline([('preprocesamiento', preprocessing),
                 ('regresor', LinearRegression())])

Luego, sólo debemos entrenar el pipeline completo

In [14]:
pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocesamiento', ...), ('regresor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('standard', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differen

In [15]:
y_pred = pipe.predict(X_test)
y_pred.shape

(120,)

In [16]:
y_test.shape

(120,)

In [17]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, median_absolute_error, root_mean_squared_error

def evaluate(y_test, y_pred):

    print('MSE:', mean_squared_error(y_test, y_pred))
    print('RMSE:', root_mean_squared_error(y_test, y_pred))
    print('MAE:', mean_absolute_error(y_test, y_pred))
    print('MedAE:', median_absolute_error(y_test, y_pred))
    print('R²:', r2_score(y_test, y_pred))
    
    
evaluate(y_test, y_pred)

MSE: 10.558109353726449
RMSE: 3.2493244457465997
MAE: 2.514332522143875
MedAE: 1.967649907035371
R²: 0.8353823529044118


In [18]:
print('Coeficientes de la Regresión (w_i) por Atributo:\n\n', pipe[-1].coef_.round(3))

Coeficientes de la Regresión (w_i) por Atributo:

 [ 2.763 -5.584  0.52   2.681  2.808 -1.358  0.805]


In [19]:
pipe[-1].intercept_ 

np.float64(20.596140928901246)

### Regularización

Regularización es un conjunto de térnicas que fuerzan a la regresión a generar modelos no tan complejos y así evitar el overfitting.

<div align='center'>
<img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/20_regresion_lineal/tipos_fit.png?raw=true' width=800/>
</div>

La idea por detrás es simple: para crear modelos regularizados se agregan distintas penalizaciones a la función objetivo (MSE). Estas penalizaciones aumentan el valor de MSE a medida que el modelo se hace más complejo (i.e., a medida que los parámetros $w_i$ se hacen más grandes).

Las penalizaciones más comunes consisten en agregar a MSE el cálculo de una norma

- **L2** sobre los parámetros, también conocida como *Rigde*,  
- **L1** sobre los parámetros, también conocida como *Lasso*
- **Elastic-Net**, la cuál es una combinación de las anteriores.


La idea fundamental de esto es que ningún parámetro sea mucho más grande que el resto. Al penalizar los parámetros grandes, este tipo de técnicas forzará que todos los $w_i$ sean los más cercanos a 0.


#### Ridge 

Como dijimos anteriormente, Ridge agrega una penalización L2 sobre la función objetivo:

$MSE = \frac{1}{N}\sum_{i=1\dots n} (y_i - f_{\mathbf{w}, b} (x_i))^2  + \alpha ||w||^2$ 

en donde $||w||$ es la norma se define como $||w||^2 = \sum_{i=1}^D (w_i)^2$ y $\alpha$ es un hiperparámetro que controla la importancia de la regularización. 

Mientras menor sea $\alpha$, menor efecto tendrá la regularización. Por otra parte, mientras mayor sea, es más posible que la regresión no sea capaz de aprender lo suficientemente bien, lo que puede llevar a un *underfitting*

L2 en general tiende a dar mejores resultados que una regresión normal cuando hay muchos atributos.

In [20]:
from sklearn.linear_model import Ridge

rigde_pipe = Pipeline([('preprocesamiento', preprocessing) , 
                       ('regresor', Ridge())])
rigde_pipe.fit(X_train, y_train)

ridge_y_pred = rigde_pipe.predict(X_test)

evaluate(y_test, ridge_y_pred)


MSE: 10.595372578009664
RMSE: 3.2550533909614545
MAE: 2.515812056705651
MedAE: 1.912894785203937
R²: 0.834801360219151


#### Lasso

Por otra parte, Lasso agrega una penalización L1 sobre la función objetivo:


$MSE = \frac{1}{N}\sum_{i=1\dots n} (y_i - f_{\mathbf{w}, b} (x_i))^2  + \alpha |w|$

en donde $|w|$ es la suma de los valores absolutos de los parámetros y se define como $|w| = \sum_{i=1}^D |w_i|$ y $\alpha$ es un hiperparámetro que controla la importancia de la regularización. 

Al igual que el caso anterior, mientras menor sea $\alpha$, menor efecto tendrá la regularización. Por otra parte, mientras mayor sea, es más posible que la regresión no sea capaz de aprender lo suficientemente bien, lo que puede llevar a un *underfitting*

Lasso produce modelos sparse, es decir, modelos que muchos de las pendientes de los atributos es igual a 0.
En términos prácticos, produce una selección de atributos al conservar solo los atributos más relevantes para predecir.
Esto permite tener interpretabilidad del modelo al saber cuales son las variables conservadas y cuales son las descartadas.

In [21]:
from sklearn.linear_model import Lasso

lasso_pipe = Pipeline([('preprocesamiento', preprocessing) , 
                       ('regresor', Lasso(alpha=0.1))])

lasso_pipe.fit(X_train, y_train)

lasso_y_pred = lasso_pipe.predict(X_test)
evaluate(y_test, lasso_y_pred)

MSE: 11.052430766886612
RMSE: 3.3245196294933517
MAE: 2.542014336473677
MedAE: 2.1989653842169155
R²: 0.827675099151194


In [22]:
print('Coeficientes de la regresión (w_i) usando Lasso por Atributo:\n\n', lasso_pipe[-1].coef_.round(3))

Coeficientes de la regresión (w_i) usando Lasso por Atributo:

 [ 0.    -4.382  0.032  0.895  1.056 -0.747  0.756]


#### Elastic-Net

Por último, Elastic-Net combina ambas penalizaciones utilizando un coeficiente $\rho$ que las pondera.


$MSE = \frac{1}{N}\sum_{i=1\dots n} (y_i - f_{\mathbf{w}, b} (x_i))^2  + \alpha \rho |w| + (1-\rho) \alpha ||w||_2$

In [23]:
from sklearn.linear_model import ElasticNet

en_pipe = Pipeline([('preprocesamiento', preprocessing) ,
                    ('regresor', ElasticNet(alpha=0.01))])
en_pipe.fit(X_train, y_train)

en_y_pred = en_pipe.predict(X_test)

evaluate(y_test, en_y_pred)

MSE: 10.621015187548352
RMSE: 3.2589899029528078
MAE: 2.5154067027479545
MedAE: 1.9616889180374386
R²: 0.8344015513228584


### Otros Modelos


Scikit-learn implementa una gran variedad de [modelos lineales](https://scikit-learn.org/stable/modules/linear_model.html#multi-task-elastic-net) que mejoran la regresión lineal simple. Entre estos podemos encontrar:

- [Bayesian Regression](https://scikit-learn.org/stable/modules/linear_model.html#bayesian-regression) por ejemplo, el cual en el entrenamiento del modelo se estiman los parámetros de la regualización L2.
- [Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression) la cual adapta una regresión para poder clasificar, todo a través de adaptar la regresión para calcular una probabilidad de que un ejemplo pertenezca a cierta clase (usando la función sigmoidal).
- Etc...

> **Pregunta ❓:** Cuántos modelos hemos entrenado ya? Cual es mejor? Cómo veo sus métricas?

## MLFlow 🧪

En proyectos de Machine Learning es común entrenar múltiples modelos con distintos parámetros, datos o algoritmos. En este proceso pronto pueden encontrarse optimizando un modelo probando muchísimos algoritmos, hiperparámetros y setups diferentes. Así, se vuelve un dolor de cabeza rastrear los diferentes experimentos y fácilmente se puede perder el modelo con las características deseadas. Sin un buen sistema de registro de experimentos, rápidamente se vuelve difícil responder preguntas como:

- ¿Qué modelo funcionó mejor?
- ¿Qué parámetros se usaron?
- ¿Se puede reproducir el resultado?

**MLflow** es una herramienta que permite registrar y organizar experimentos de Machine Learning, guardando:

- Parámetros del modelo
- Métricas de desempeño
- Modelos entrenados
- Archivos relevantes

De esta forma, cada entrenamiento queda documentado y es posible comparar resultados de manera sistemática. Esto no solo permite loggear modelos, sino que también el pipeline completo

🎯 Idea clave
> Cada vez que entrenas un modelo, deberías registrar qué hiciste y qué resultados obtuviste.

In [ ]:
# Instalamos
!uv add mlflow
!uv add "protobuf<4"
!uv pip install "setuptools<70"

In [ ]:
import mlflow
import mlflow.sklearn

# 1. Definir experimento
mlflow.set_experiment("regresion_mpg")

# 2. Parámetros del modelo
alpha = 0.1

# 3. Iniciar run
with mlflow.start_run(run_name="ridge_alpha_0.1"):

    # Definir el pipeline
    pipe = Pipeline([
        ('preprocesamiento', preprocessing), 
        ('regresor', Ridge(alpha=alpha))
    ])

    # Entrenar el pipeline completo
    pipe.fit(X_train, y_train)

    # Predicciones
    y_pred = pipe.predict(X_test)

    # Métrica
    r2 = r2_score(y_test, y_pred)

    # 4. Loggear parámetros
    mlflow.log_param("alpha", alpha)

    # 5. Loggear métricas
    mlflow.log_metric("r2_score", r2)

    # 6. Loggear pipeline como un modelo
    mlflow.sklearn.log_model(pipe, "pipeline_ridge")

    print(f"R2 Score: {r2}")

2026/04/26 19:11:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


R2 Score: 0.8353362233499575


#### Modelos registrados

Para ver los modelos registrados, simplemente podemos ejecutar en una terminal

```bash
mlflow ui
```

Y luego visitar http://localhost:5000

### Loggeando múltiples pipelines

Como pueden ver, la definición de qué es un parámetro y qué es una métrica en MLFlow es bastante flexible. Básicamente cualquier variación que podamos hacer a un modelo puede ser considerada un hiperparámetro, incluso si no es un hiperparámetro formal dado al inicializar el modelo de sklearn. Esto, sumado a que MLFlow permite loggear tanto modelos como pipelines completos, nos permite hacer experimentos extensivos y completamente reproducibles

> **Buena práctica 🎯:** Loggear pipelines completos no solamente es permitido, sino que **recomendado**, ya que aumenta la reproducibilidad de los experimentos

In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# 1. Definir experimento
mlflow.set_experiment("regresion_mpg")

# 2. Definir hiperparámetros
escaladores = {
    "standard": StandardScaler(),
    "minmax": MinMaxScaler(),
    "robust": RobustScaler()
}
modelos = {
    "regresion_lineal_simple": LinearRegression(),
    "regresion_ridge": Ridge(),
    "lasso": Lasso(alpha=0.1),
    "elastic_net": ElasticNet(alpha=0.01)
}

# 3. Iteramos corrida con cada hiperparámetro
for nombre_escalador, escalador in escaladores.items():
    for tipo_modelo, modelo in modelos.items():
        run_name = f"{tipo_modelo}-{nombre_escalador}"
        with mlflow.start_run(run_name=run_name):

            # Definir el column transformer
            preprocessing = ColumnTransformer(
                [
                    ("escalador", escalador, ["displacement", "weight", "acceleration"]),
                    ("ohe", OneHotEncoder(drop='first'), ["origin"]),
                    ("ordinal", OrdinalEncoder(), ["cylinders", "model year"]),
                ]
            )

            # Definimos el pipeline-
            pipe = Pipeline([
                ('preprocesamiento', preprocessing), 
                ('regresor', modelo)
            ])

            # Entrenar el pipeline completo
            pipe.fit(X_train, y_train)

            # Predicciones
            y_pred = pipe.predict(X_test)

            # 4. Loggear parámetros
            mlflow.log_param("escalador", nombre_escalador)
            mlflow.log_param("tipo_modelo", tipo_modelo)

            # 5. Loggear métricas
            mlflow.log_metric('MSE', mean_squared_error(y_test, y_pred))
            mlflow.log_metric('RMSE', root_mean_squared_error(y_test, y_pred))
            mlflow.log_metric('MAE', mean_absolute_error(y_test, y_pred))
            mlflow.log_metric('MedAE', median_absolute_error(y_test, y_pred))
            mlflow.log_metric('R²', r2_score(y_test, y_pred))

            # 6. Loggear pipeline como un modelo
            mlflow.sklearn.log_model(pipe, f"pipeline_{tipo_modelo}")



2026/04/26 19:41:39 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/26 19:41:43 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/26 19:41:48 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/26 19:41:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/26 19:41:58 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/26 19:42:03 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be adde

Luego de ejecutar esto, pueden visitar http://localhost:5000 para revisar cuales son las métricas de los modelos entrenados y cuales tienen mejor desempeño según la métrica esperada. Podemos cargar este modelo de la siguiente forma

In [34]:
loaded_model = mlflow.sklearn.load_model("runs:/4691a87843734ac6b9ce30d9845c1601/pipeline_regresion_lineal_simple")
preds = loaded_model.predict(X_test)
preds

array([26.37652777, 21.9285936 , 31.59074914, 35.89885131, 19.91659982,
       23.76794371, 12.06235417, 13.91217061, 12.74191894, 31.49236887,
       29.30340534, 23.06933445, 23.82279496, 13.29350554, 27.09361425,
       27.22530931, 29.8609291 , 12.78297415, 31.5916736 , 11.23967273,
       22.35051544, 14.94626078, 13.95372773, 19.59818073, 24.86883386,
       35.94492236, 31.25727543, 10.86343976, 30.30123946, 16.01616094,
       32.55108618, 34.40515718, 11.95116402, 29.19839775,  7.09057909,
       33.2816863 ,  9.95151174, 14.15152723, 11.31059275, 24.43277476,
       33.61232951, 26.31172937, 17.46305429, 25.83741444, 25.91594566,
       26.47464636, 25.50962413, 25.85412873, 14.98932662, 27.5439335 ,
       32.19670477, 17.20772063, 16.40940716, 20.00782003, 29.81365591,
       34.51825696, 28.80090723, 17.13735083, 27.14075198, 21.23191522,
       31.06249589, 20.17991956, 28.00768746, 13.76145276, 24.463601  ,
       24.10339241, 20.70804862, 30.2927674 , 11.44972985, 20.36